# Ham Veri Keşfi – NYC Yellow Taxi 2024 (Ocak–Mart)
Bronze katmanını yazmadan önce ham dosyaları tanımak için ilk inceleme.
Cevap aranan sorular: kolonlar ve tipleri, aylar arası schema farkı, ay bazında satır sayısı ve tarih aralığı, null oranları, zone lookup yapısı.

## 1. Spark session ve dosya path'leri
Spark'ı başlatır ve her ayın dosya path'ini bir dictionary'de toplar.

- `SparkSession.builder ... .getOrCreate()`: Spark'a giriş noktası. Açık bir session varsa onu kullanır, yoksa yenisini açar.
- `.master("local[*]")`: Local mode; makinedeki tüm CPU çekirdekleri kullanılır.
- `RAW_PATH`: Ham verinin container içindeki tam path'i.
- `MONTHS`: İncelenecek aylar.
- `trip_files_by_month`: Boş bir dictionary. `for` döngüsü her ay için bir kayıt ekler: key = ay numarası, value = dosya path'i.
- `{month:02d}`: Ayı 2 haneli yazar (1 → 01).

In [25]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("raw-exploration")
    .master("local[*]")  # local mode, tüm CPU çekirdekleri
    .getOrCreate()
)

# Container içindeki ham veri klasörü
RAW_PATH = "/home/jovyan/work/data/raw"
MONTHS = [1, 2, 3]

# Ay numarası -> dosya path'i
trip_files_by_month = {}
for month in MONTHS:
    file_path = f"{RAW_PATH}/yellow_tripdata_2024-{month:02d}.parquet"
    trip_files_by_month[month] = file_path

print(trip_files_by_month)

{1: '/home/jovyan/work/data/raw/yellow_tripdata_2024-01.parquet', 2: '/home/jovyan/work/data/raw/yellow_tripdata_2024-02.parquet', 3: '/home/jovyan/work/data/raw/yellow_tripdata_2024-03.parquet'}


## 2. Ocak ayının schema'sı ve örnek satırlar
Ocak dosyasını okur; kolonları, ilk satırları ve satır sayısını gösterir. Ocak, diğer aylar için referans olacak.

- `spark.read.parquet(...)`: Parquet dosyasını DataFrame olarak okur (`df_jan`).
- `printSchema()`: Kolon adlarını ve tiplerini ağaç şeklinde gösterir.
- `show(5, truncate=False)`: İlk 5 satırı gösterir; `truncate=False` uzun değerlerin kesilmemesi için.
- `count()`: Satır sayısı. Bir action'dır, yani gerçek hesaplamayı tetikler.

In [27]:
df_jan = spark.read.parquet(trip_files_by_month[1])

df_jan.printSchema()
df_jan.show(5, truncate=False)

row_count_jan = df_jan.count()
print("January row count:", row_count_jan)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------

## 3. Aylar arası schema karşılaştırması
Her ayın kolon adlarını ve tiplerini Ocak ile karşılaştırır, farkları listeler. Bronze'da schema'yı tek tipe standardize ederken bu bilgiyi kullanacağız.

- `schema_to_dict(df)`: Yardımcı function. DataFrame'in schema'sını `{kolon adı: tip}` dictionary'sine çevirir. Aynı işi her ay için tekrar yazmamak için function yapıldı.
- `df.schema.fields`: Kolonların listesi; her `field`'ın `name` (ad) ve `dataType` (tip) bilgisi var.
- `schema_jan`: Ocak'ın kolon → tip eşlemesi (referans).
- Dış `for` döngüsü: Her ayı okur ve `schema_current`'ı üretir.
- **Kontrol 1:** Ocak'taki her kolon bu ayda var mı, tipi aynı mı? `.get(column_name, "(missing)")` kolon yoksa hata vermek yerine `"(missing)"` döner.
- **Kontrol 2:** Bu ayda olup Ocak'ta olmayan kolon var mı? (`not in`)
- `differences`: Bulunan farkların listesi. Boşsa "same as January" yazar, doluysa farkları tek tek yazar.

In [30]:
def schema_to_dict(df):
    """DataFrame'in schema'sını kolon adı -> tip dictionary'si olarak döner."""
    column_types = {}
    for field in df.schema.fields:
        column_types[field.name] = field.dataType.simpleString()
    return column_types


# Referans schema: Ocak
schema_jan = schema_to_dict(df_jan)

for month, file_path in trip_files_by_month.items():
    df_month = spark.read.parquet(file_path)
    schema_current = schema_to_dict(df_month)

    differences = []

    # Kontrol 1: Ocak'taki kolon bu ayda eksik mi ya da tipi farklı mı?
    for column_name, type_jan in schema_jan.items():
        type_current = schema_current.get(column_name, "(missing)")
        if type_current != type_jan:
            differences.append(f"{column_name}: {type_jan} -> {type_current}")

    # Kontrol 2: Sadece bu ayda olan kolonlar
    for column_name, type_current in schema_current.items():
        if column_name not in schema_jan:
            differences.append(f"{column_name}: (missing) -> {type_current}")

    # Bu ayın sonucunu yazdır
    if len(differences) == 0:
        print(f"Month {month}: same as January")
    else:
        print(f"Month {month}: {len(differences)} difference(s)")
        for difference in differences:
            print("   ", difference)

Month 1: same as January
Month 2: same as January
Month 3: same as January


## 4. Dosya bazında satır sayısı ve pickup tarih aralığı
Üç ayı tek DataFrame olarak okur; her dosya için satır sayısını ve en erken / en geç pickup zamanını gösterir. Ay dışına taşan tarihler silver'daki temizlik kuralının girdisi olacak.

- `yellow_tripdata_2024-*.parquet`: Glob pattern. `*` "herhangi bir metin" demek, böylece üç dosya birden okunur (`df_trips`).
- `_metadata.file_name`: Spark'ın gizli kolonu; satırın hangi dosyadan geldiğini söyler. `withColumn` ile `source_file` adında görünür bir kolona çevriliyor. Bronze'daki `_source_file` kolonu da bununla doldurulacak.
- `groupBy("source_file").agg(...)`: Dosya bazında gruplar. `count("*")` satır sayısını, `min` / `max` en erken ve en geç pickup zamanını verir.
- `.alias(...)`: Sonuç kolonuna isim verir.
- `orderBy`: Sonucu dosya adına göre sıralar.

In [31]:
# Üç ayı glob pattern ile tek seferde oku (* = herhangi bir metin)
df_trips = spark.read.parquet(f"{RAW_PATH}/yellow_tripdata_2024-*.parquet")

# _metadata.file_name gizli bir kolon: satırın geldiği dosya
df_trips_with_file = df_trips.withColumn("source_file", F.col("_metadata.file_name"))

df_file_summary = (
    df_trips_with_file
    .groupBy("source_file")
    .agg(
        F.count("*").alias("row_count"),
        F.min("tpep_pickup_datetime").alias("min_pickup"),
        F.max("tpep_pickup_datetime").alias("max_pickup"),
    )
    .orderBy("source_file")
)

df_file_summary.show(truncate=False)

+-------------------------------+---------+-------------------+-------------------+
|source_file                    |row_count|min_pickup         |max_pickup         |
+-------------------------------+---------+-------------------+-------------------+
|yellow_tripdata_2024-01.parquet|2964624  |2002-12-31 22:59:39|2024-02-01 00:01:15|
|yellow_tripdata_2024-02.parquet|3007526  |2008-12-31 22:52:49|2024-03-01 00:01:37|
|yellow_tripdata_2024-03.parquet|3582628  |2002-12-31 22:17:10|2024-04-01 00:34:55|
+-------------------------------+---------+-------------------+-------------------+



## 5. Kolon bazında null oranı (%)
Her kolonda değerlerin yüzde kaçının null olduğunu hesaplar.

- `for` döngüsü: Her kolon için bir hesaplama ifadesi (expression) üretip `null_pct_expressions` listesine ekler. Bu aşamada henüz hesaplama yapılmaz.
- `isNull()`: Her satır için true/false.
- `cast("int")`: true → 1, false → 0.
- `F.avg(...)`: 1/0 değerlerinin ortalaması = null oranı (ör. 0.05).
- `* 100` ve `F.round(..., 2)`: Yüzdeye çevirir, 2 ondalığa yuvarlar.
- `df_trips.select(null_pct_expressions)`: Tüm kolonlar tek bir `select` ile hesaplanır; veri tek seferde taranır.
- `show(vertical=True)`: Tek satırlık çok kolonlu sonucu alt alta gösterir.

In [33]:
# Her kolon için bir null yüzdesi ifadesi oluştur
null_pct_expressions = []
for column_name in df_trips.columns:
    # isNull() true/false verir; int'e çevirince 1/0; 1/0 ortalaması null oranıdır
    is_null_as_int = F.col(column_name).isNull().cast("int")
    null_pct = F.round(F.avg(is_null_as_int) * 100, 2).alias(column_name)
    null_pct_expressions.append(null_pct)

# Tek select ile tüm kolonlar, veri bir kez taranır
df_null_pct = df_trips.select(null_pct_expressions)

# vertical=True: her kolon ayrı satırda
df_null_pct.show(vertical=True)

-RECORD 0---------------------
 VendorID              | 0.0  
 tpep_pickup_datetime  | 0.0  
 tpep_dropoff_datetime | 0.0  
 passenger_count       | 7.87 
 trip_distance         | 0.0  
 RatecodeID            | 7.87 
 store_and_fwd_flag    | 7.87 
 PULocationID          | 0.0  
 DOLocationID          | 0.0  
 payment_type          | 0.0  
 fare_amount           | 0.0  
 extra                 | 0.0  
 mta_tax               | 0.0  
 tip_amount            | 0.0  
 tolls_amount          | 0.0  
 improvement_surcharge | 0.0  
 total_amount          | 0.0  
 congestion_surcharge  | 7.87 
 Airport_fee           | 7.87 



## 6. Taxi zone lookup
Zone lookup CSV'sini okur ve yapısını gösterir. Silver'da `PULocationID` / `DOLocationID` numaralarını zone ve borough isimlerine çevirmek için join edilecek.

- `option("header", True)`: İlk satır kolon adlarıdır.
- `option("inferSchema", True)`: Kolon tiplerini Spark tahmin etsin; aksi halde hepsi string olur.
- `.csv(...)`: CSV'yi DataFrame olarak okur (`df_zones`).
- `count()` ve `show(10)`: Zone sayısı (beklenen ~265) ve ilk 10 satır.

In [35]:
df_zones = (
    spark.read
    .option("header", True)       # ilk satır kolon adları
    .option("inferSchema", True)  # tipleri Spark tahmin etsin
    .csv(f"{RAW_PATH}/taxi_zone_lookup.csv")
)

df_zones.printSchema()

zone_count = df_zones.count()
print("Zone count:", zone_count)

df_zones.show(10, truncate=False)

root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: string (nullable = true)
 |-- service_zone: string (nullable = true)

Zone count: 265
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park          

## 7. Null grubu kontrolü
Hücre 5'te beş kolonun null oranı aynı çıktı (%7,87). Bu hücre, null'ların aynı satırlarda olup olmadığını kontrol eder.

- `null_group_columns`: Kontrol edilecek beş kolon.
- `all_null_condition`: "Beş kolonun hepsi null" koşulu. `F.lit(True)` ile başlar, döngüde her kolon `&` (and) ile eklenir.
- `any_null_condition`: "En az biri null" koşulu. `F.lit(False)` ile başlar, döngüde her kolon `|` (or) ile eklenir.
- `F.lit(...)`: Sabit değer üretir. Başlangıç değeri sonucu değiştirmez, sadece döngüye bir başlangıç noktası verir.
- İki sayı eşitse null'lar aynı satırlardadır: bir kolon null ise diğer dördü de null.
- Son adım: Bu satırların `payment_type` dağılımı. Null'ların belirli bir ödeme tipine bağlı olup olmadığını gösterir.

In [36]:
null_group_columns = [
    "passenger_count",
    "RatecodeID",
    "store_and_fwd_flag",
    "congestion_surcharge",
    "Airport_fee",
]

# Koşulları kolon kolon adım adım kur
all_null_condition = F.lit(True)    # hepsi null
any_null_condition = F.lit(False)   # en az biri null
for column_name in null_group_columns:
    all_null_condition = all_null_condition & F.col(column_name).isNull()
    any_null_condition = any_null_condition | F.col(column_name).isNull()

df_all_null = df_trips.filter(all_null_condition)
df_any_null = df_trips.filter(any_null_condition)

all_null_count = df_all_null.count()
any_null_count = df_any_null.count()

print("Rows where all 5 columns are null:", all_null_count)
print("Rows where at least 1 column is null:", any_null_count)

if all_null_count == any_null_count:
    print("Result: nulls are in the same rows")
else:
    print("Result: nulls are NOT always in the same rows")

# Null satırların ödeme tipi dağılımı
df_null_by_payment = (
    df_all_null
    .groupBy("payment_type")
    .count()
    .orderBy("payment_type")
)
df_null_by_payment.show()

Rows where all 5 columns are null: 751962
Rows where at least 1 column is null: 751962
Result: nulls are in the same rows
+------------+------+
|payment_type| count|
+------------+------+
|           0|751962|
+------------+------+

